In [1]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ============================================================
# 17 - HYPERPARAMETER OPTIMIZATION
# CELL 0: INITIALIZATION
# ============================================================

import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

RANDOM_SEED = 42

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3

N_TRIALS = 12

PRIMARY_METRIC = "Exact_Match_Accuracy"

MODEL_NAMES = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí",
    "Hóa học",
    "Sinh học",
    "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học",
    "Công nghệ",
    "Âm nhạc",
    "Mĩ thuật"
]

lib.np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

split_dir = ROOT_DIR / "data/data-splits"

fold_dir = split_dir / "cv_folds_v1"

output_dir = ROOT_DIR / (
    "experiments/v2_optimization/"
    "06_hyperparameter_tuning/results"
)

table_dir = output_dir / "tables"
config_dir = output_dir / "configs"
study_dir = output_dir / "studies"
figure_dir = output_dir / "figures"

for directory in [
    table_dir,
    config_dir,
    study_dir,
    figure_dir
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# DEVELOPMENT DATA
# ------------------------------------------------------------

df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig",
    dtype={id_col: str}
)

df_dev[id_col] = (
    df_dev[id_col].str.strip()
)

score_cols = [
    col for col in df_dev.columns
    if col.startswith((
        "L6_", "L7_", "L8_", "L9_"
    ))
    and col.endswith((
        "_hki", "_hkii", "_ca_nam"
    ))
]

X = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()

assert X.shape == (9000, 96)
assert y.shape == (9000, 9)

assert df_dev[id_col].notna().all()
assert df_dev[id_col].is_unique

assert X.notna().all().all()
assert lib.np.isfinite(
    X.to_numpy(dtype=float)
).all()

assert y.isin([0, 1]).all().all()
assert y.sum(axis=1).eq(4).all()

# ------------------------------------------------------------
# FIXED OUTER FOLDS FROM V1
# ------------------------------------------------------------

id_to_position = {
    student_id: i
    for i, student_id
    in enumerate(df_dev[id_col])
}

all_dev_ids = set(df_dev[id_col])

outer_folds = []
fold_records = []

for fold in range(1, N_OUTER_FOLDS + 1):

    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv",
        encoding="utf-8-sig",
        dtype={id_col: str}
    )[id_col].str.strip()

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv",
        encoding="utf-8-sig",
        dtype={id_col: str}
    )[id_col].str.strip()

    assert train_ids.notna().all()
    assert val_ids.notna().all()
    assert train_ids.is_unique
    assert val_ids.is_unique

    train_set = set(train_ids)
    val_set = set(val_ids)

    assert train_set.isdisjoint(val_set)
    assert train_set | val_set == all_dev_ids

    train_idx = lib.np.array(
        sorted(
            id_to_position[sid]
            for sid in train_set
        ),
        dtype=int
    )

    val_idx = lib.np.array(
        sorted(
            id_to_position[sid]
            for sid in val_set
        ),
        dtype=int
    )

    outer_folds.append({
        "Fold": fold,
        "train_idx": train_idx,
        "val_idx": val_idx
    })

    fold_records.append({
        "Fold": fold,
        "Outer_Train": len(train_idx),
        "Outer_Validation": len(val_idx),
        "Total": len(train_idx) + len(val_idx)
    })

# Đảm bảo mỗi học sinh vào Outer Validation đúng một lần
all_val_idx = lib.np.concatenate([
    item["val_idx"]
    for item in outer_folds
])

assert lib.np.array_equal(
    lib.np.sort(all_val_idx),
    lib.np.arange(len(df_dev))
)

fold_report = lib.pd.DataFrame(fold_records)

print("=" * 65)
print("NESTED CROSS-VALIDATION INITIALIZATION")
print("=" * 65)

print("Development:", df_dev.shape)
print("Features:", X.shape[1])
print("Targets:", y.shape[1])

print("\nOUTER FOLDS")
display(fold_report)

print("\nOptuna Trials:", N_TRIALS)
print("Inner Folds:", N_INNER_FOLDS)

print("\nPASS: Dữ liệu và Outer CV hợp lệ.")

NESTED CROSS-VALIDATION INITIALIZATION
Development: (9000, 106)
Features: 96
Targets: 9

OUTER FOLDS


,Fold,Outer_Train,Outer_Validation,Total
0,1,7203,1797,9000
1,2,7201,1799,9000
2,3,7190,1810,9000
3,4,7220,1780,9000
4,5,7186,1814,9000



Optuna Trials: 12
Inner Folds: 3

PASS: Dữ liệu và Outer CV hợp lệ.


In [2]:
# ============================================================
# CELL 1: RANDOM FOREST SEARCH SPACE
# ============================================================

def suggest_rf_params(trial):

    return {
        "n_estimators": trial.suggest_categorical(
            "n_estimators",
            [100, 200, 300]
        ),
        "max_depth": trial.suggest_categorical(
            "max_depth",
            [None, 10, 20]
        ),
        "min_samples_leaf": trial.suggest_categorical(
            "min_samples_leaf",
            [1, 2, 4]
        ),
        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", 0.7]
        )
    }


print("RANDOM FOREST SEARCH SPACE")

print({
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.7]
})

RANDOM FOREST SEARCH SPACE
{'n_estimators': [100, 200, 300], 'max_depth': [None, 10, 20], 'min_samples_leaf': [1, 2, 4], 'max_features': ['sqrt', 0.7]}


In [3]:
# ============================================================
# CELL 2: XGBOOST SEARCH SPACE
# ============================================================

def suggest_xgb_params(trial):

    return {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            150,
            350,
            step=50
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            7
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.03,
            0.15,
            log=True
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.75,
            1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.75,
            1.0
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10.0,
            log=True
        )
    }


print("XGBOOST SEARCH SPACE")

print({
    "n_estimators": "150 - 350",
    "max_depth": "3 - 7",
    "learning_rate": "0.03 - 0.15",
    "subsample": "0.75 - 1.0",
    "colsample_bytree": "0.75 - 1.0",
    "reg_lambda": "0.1 - 10.0"
})

XGBOOST SEARCH SPACE
{'n_estimators': '150 - 350', 'max_depth': '3 - 7', 'learning_rate': '0.03 - 0.15', 'subsample': '0.75 - 1.0', 'colsample_bytree': '0.75 - 1.0', 'reg_lambda': '0.1 - 10.0'}


In [4]:
# ============================================================
# CELL 3: CATBOOST SEARCH SPACE
# ============================================================

def suggest_cat_params(trial):

    return {
        "iterations": trial.suggest_int(
            "iterations",
            250,
            600,
            step=50
        ),
        "depth": trial.suggest_int(
            "depth",
            4,
            8
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.025,
            0.12,
            log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1.0,
            10.0,
            log=True
        )
    }


print("CATBOOST SEARCH SPACE")

print({
    "iterations": "250 - 600",
    "depth": "4 - 8",
    "learning_rate": "0.025 - 0.12",
    "l2_leaf_reg": "1.0 - 10.0"
})

CATBOOST SEARCH SPACE
{'iterations': '250 - 600', 'depth': '4 - 8', 'learning_rate': '0.025 - 0.12', 'l2_leaf_reg': '1.0 - 10.0'}


In [5]:
# ============================================================
# CELL 4: MODEL FACTORY & TOP-4 PREDICTION
# ============================================================

def create_model(model_name, params):

    params = params.copy()

    if model_name == "Random Forest":

        base_model = RandomForestClassifier(
            **params,
            criterion="gini",
            min_samples_split=2,
            bootstrap=True,
            class_weight=None,
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

    elif model_name == "XGBoost":

        base_model = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

    elif model_name == "CatBoost":

        base_model = CatBoostClassifier(
            **params,
            loss_function="Logloss",
            random_seed=RANDOM_SEED,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return MultiOutputClassifier(
        estimator=base_model,
        n_jobs=1
    )


def predict_positive_proba(model, X_data):

    probabilities = []

    for estimator in model.estimators_:

        classes = list(estimator.classes_)

        assert 1 in classes, (
            "Bộ phân loại không có lớp dương."
        )

        positive_index = classes.index(1)

        prob = estimator.predict_proba(
            X_data
        )[:, positive_index]

        probabilities.append(prob)

    return lib.np.column_stack(
        probabilities
    )


def predict_top4(probabilities):

    assert probabilities.ndim == 2
    assert probabilities.shape[1] == 9

    y_pred = lib.np.zeros_like(
        probabilities,
        dtype=int
    )

    top4_idx = lib.np.argsort(
        -probabilities,
        axis=1,
        kind="stable"
    )[:, :4]

    rows = lib.np.arange(
        len(probabilities)
    )[:, None]

    y_pred[rows, top4_idx] = 1

    assert y_pred.sum(axis=1).min() == 4
    assert y_pred.sum(axis=1).max() == 4

    return y_pred


def suggest_model_params(trial, model_name):

    if model_name == "Random Forest":
        return suggest_rf_params(trial)

    if model_name == "XGBoost":
        return suggest_xgb_params(trial)

    if model_name == "CatBoost":
        return suggest_cat_params(trial)

    raise ValueError(model_name)


print("PASS: Đã khởi tạo Model Factory.")
print("PASS: Quy tắc dự đoán Top-4 đã sẵn sàng.")

PASS: Đã khởi tạo Model Factory.
PASS: Quy tắc dự đoán Top-4 đã sẵn sàng.


In [6]:
# ============================================================
# CELL 5: FIXED INNER CROSS-VALIDATION
# ============================================================

def create_inner_folds(
    outer_train_idx,
    n_splits=3
):

    inner_cv = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    # Vị trí tương đối trong Outer Training
    relative_indices = lib.np.arange(
        len(outer_train_idx)
    )

    inner_folds = []

    for inner_fold, (
        inner_train_rel,
        inner_val_rel
    ) in enumerate(
        inner_cv.split(relative_indices),
        start=1
    ):

        # Đổi về vị trí tuyệt đối trong Development
        inner_train_idx = outer_train_idx[
            inner_train_rel
        ]

        inner_val_idx = outer_train_idx[
            inner_val_rel
        ]

        assert len(
            lib.np.intersect1d(
                inner_train_idx,
                inner_val_idx
            )
        ) == 0

        inner_folds.append({
            "Inner_Fold": inner_fold,
            "train_idx": inner_train_idx,
            "val_idx": inner_val_idx
        })

    return inner_folds


# ------------------------------------------------------------
# TẠO INNER FOLDS CHO CẢ 5 OUTER FOLDS
# ------------------------------------------------------------

inner_folds_by_outer = {}
inner_records = []

for outer_info in outer_folds:

    outer_fold = outer_info["Fold"]

    outer_train_idx = outer_info["train_idx"]
    outer_val_idx = outer_info["val_idx"]

    inner_folds = create_inner_folds(
        outer_train_idx,
        n_splits=N_INNER_FOLDS
    )

    inner_folds_by_outer[outer_fold] = inner_folds

    inner_validation_positions = []

    for inner_info in inner_folds:

        inner_train_idx = inner_info["train_idx"]
        inner_val_idx = inner_info["val_idx"]

        # Inner CV không được chứa Outer Validation
        assert len(
            lib.np.intersect1d(
                inner_train_idx,
                outer_val_idx
            )
        ) == 0

        assert len(
            lib.np.intersect1d(
                inner_val_idx,
                outer_val_idx
            )
        ) == 0

        inner_validation_positions.extend(
            inner_val_idx.tolist()
        )

        inner_records.append({
            "Outer_Fold": outer_fold,
            "Inner_Fold": inner_info["Inner_Fold"],
            "Inner_Train": len(inner_train_idx),
            "Inner_Validation": len(inner_val_idx),
            "Outer_Validation_Leakage": 0
        })

    # Mỗi học sinh Outer Training phải nằm trong
    # một Inner Validation đúng một lần
    assert lib.np.array_equal(
        lib.np.sort(
            lib.np.array(inner_validation_positions)
        ),
        lib.np.sort(outer_train_idx)
    )


inner_report = lib.pd.DataFrame(
    inner_records
)

print("=" * 65)
print("INNER CROSS-VALIDATION VALIDATION")
print("=" * 65)

display(inner_report)

print("\nPASS: Inner CV chỉ sử dụng Outer Training.")
print("PASS: Mọi trial sử dụng cùng Inner Fold.")
print("PASS: Không có Outer Validation leakage.")

INNER CROSS-VALIDATION VALIDATION


,Outer_Fold,Inner_Fold,Inner_Train,Inner_Validation,Outer_Validation_Leakage
0,1,1,4802,2401,0
1,1,2,4802,2401,0
2,1,3,4802,2401,0
3,2,1,4800,2401,0
4,2,2,4801,2400,0
5,2,3,4801,2400,0
6,3,1,4793,2397,0
7,3,2,4793,2397,0
8,3,3,4794,2396,0
9,4,1,4813,2407,0



PASS: Inner CV chỉ sử dụng Outer Training.
PASS: Mọi trial sử dụng cùng Inner Fold.
PASS: Không có Outer Validation leakage.


In [7]:
# ============================================================
# CELL 6: OPTUNA OBJECTIVE FUNCTION
# ============================================================

def build_objective(
    model_name,
    inner_folds
):

    def objective(trial):

        params = suggest_model_params(
            trial,
            model_name
        )

        exact_scores = []
        macro_scores = []
        micro_scores = []
        hamming_scores = []

        for inner_info in inner_folds:

            inner_fold = inner_info["Inner_Fold"]

            train_idx = inner_info["train_idx"]
            val_idx = inner_info["val_idx"]

            X_train = X.iloc[train_idx]
            X_val = X.iloc[val_idx]

            y_train = y.iloc[train_idx]
            y_val = y.iloc[val_idx]

            # Kiểm tra cả hai lớp có trong Training
            # đối với mỗi nhãn nhị phân
            class_counts = y_train.sum(axis=0)

            assert (
                (class_counts > 0)
                & (class_counts < len(y_train))
            ).all(), (
                "Inner Training thiếu lớp 0 hoặc 1."
            )

            model = create_model(
                model_name,
                params
            )

            # Fit chỉ trên Inner Training
            model.fit(
                X_train,
                y_train
            )

            # Predict trên Inner Validation
            y_proba = predict_positive_proba(
                model,
                X_val
            )

            y_pred = predict_top4(
                y_proba
            )

            y_true = y_val.to_numpy(dtype=int)

            exact = accuracy_score(
                y_true,
                y_pred
            )

            macro = f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

            micro = f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            )

            hamming = hamming_loss(
                y_true,
                y_pred
            )

            exact_scores.append(exact)
            macro_scores.append(macro)
            micro_scores.append(micro)
            hamming_scores.append(hamming)

            # Báo cáo điểm từng bước để Optuna pruning
            current_mean = float(
                lib.np.mean(exact_scores)
            )

            trial.report(
                current_mean,
                step=inner_fold
            )

            if trial.should_prune():
                raise optuna.TrialPruned()

        # Ghi nhận các chỉ số phụ của trial hoàn thành
        trial.set_user_attr(
            "Macro_F1",
            float(lib.np.mean(macro_scores))
        )

        trial.set_user_attr(
            "Micro_F1",
            float(lib.np.mean(micro_scores))
        )

        trial.set_user_attr(
            "Hamming_Loss",
            float(lib.np.mean(hamming_scores))
        )

        trial.set_user_attr(
            "Inner_Exact_Std",
            float(
                lib.np.std(
                    exact_scores,
                    ddof=1
                )
            )
        )

        # Objective chính
        return float(
            lib.np.mean(exact_scores)
        )

    return objective


print("PASS: Objective tối ưu Exact-match Accuracy.")
print("PASS: Inner CV được tách khỏi Outer Validation.")

PASS: Objective tối ưu Exact-match Accuracy.
PASS: Inner CV được tách khỏi Outer Validation.


In [8]:
# ============================================================
# CELL 7: NESTED CV + OPTUNA OPTIMIZATION
# ============================================================

# Cấu hình V1 đưa vào Trial đầu tiên
V1_TRIAL_PARAMS = {
    "Random Forest": {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    "XGBoost": {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_lambda": 1.0
    },
    "CatBoost": {
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3.0
    }
}

# Nơi lưu kết quả từng Outer Fold
checkpoint_dir = (
    table_dir / "outer_checkpoints"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)


def model_slug(model_name):

    return {
        "Random Forest": "random_forest",
        "XGBoost": "xgboost",
        "CatBoost": "catboost"
    }[model_name]


def get_study(
    model_name,
    outer_fold,
    study_suffix=None
):

    slug = model_slug(model_name)

    if study_suffix is None:
        name = f"{slug}_outer_{outer_fold}"
    else:
        name = f"{slug}_{study_suffix}"

    db_path = study_dir / f"{name}.db"

    storage_url = (
        f"sqlite:///{db_path.as_posix()}"
    )

    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_SEED + outer_fold,
        n_startup_trials=4
    )

    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=2,
        interval_steps=1
    )

    study = optuna.create_study(
        study_name=name,
        storage=storage_url,
        load_if_exists=True,
        direction="maximize",
        sampler=sampler,
        pruner=pruner
    )

    # Đưa cấu hình V1 vào Trial 0
    if len(study.trials) == 0:

        study.enqueue_trial(
            V1_TRIAL_PARAMS[model_name]
        )

    return study


def optimize_study(
    study,
    model_name,
    inner_folds,
    total_trials=N_TRIALS
):

    # Chỉ thực hiện số trial còn thiếu
    remaining = max(
        0,
        total_trials - len(study.trials)
    )

    if remaining > 0:

        objective = build_objective(
            model_name,
            inner_folds
        )

        study.optimize(
            objective,
            n_trials=remaining,
            n_jobs=1,
            gc_after_trial=True,
            show_progress_bar=True
        )

    complete_trials = [
        trial
        for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
    ]

    if not complete_trials:
        raise RuntimeError(
            f"Study {study.study_name} không có trial hoàn thành."
        )

    return study.best_params, float(study.best_value)


outer_metric_records = []
outer_label_records = []
outer_prediction_frames = []
best_param_records = []


for model_name in MODEL_NAMES:

    slug = model_slug(model_name)

    for outer_info in outer_folds:

        outer_fold = outer_info["Fold"]

        print("=" * 70)
        print(
            f"MODEL: {model_name} | "
            f"OUTER FOLD: {outer_fold}"
        )
        print("=" * 70)

        result_path = checkpoint_dir / (
            f"{slug}_fold_{outer_fold}_metrics.csv"
        )

        label_path = checkpoint_dir / (
            f"{slug}_fold_{outer_fold}_labels.csv"
        )

        pred_path = checkpoint_dir / (
            f"{slug}_fold_{outer_fold}_predictions.csv"
        )

        params_path = config_dir / (
            f"{slug}_outer_{outer_fold}_best.json"
        )

        # ----------------------------------------------------
        # OPTUNA STUDY
        # ----------------------------------------------------

        study = get_study(
            model_name,
            outer_fold
        )

        inner_folds = inner_folds_by_outer[
            outer_fold
        ]

        best_params, best_inner_score = optimize_study(
            study,
            model_name,
            inner_folds
        )

        # Lưu toàn bộ Trial
        trials_df = study.trials_dataframe()

        trials_df.to_csv(
            table_dir / (
                f"{slug}_outer_{outer_fold}_trials.csv"
            ),
            index=False,
            encoding="utf-8-sig"
        )

        # ----------------------------------------------------
        # LƯU CẤU HÌNH TỐT NHẤT
        # ----------------------------------------------------

        best_info = {
            "Model": model_name,
            "Outer_Fold": outer_fold,
            "Best_Inner_Exact": best_inner_score,
            "Best_Params": best_params,
            "N_Trials": len(study.trials)
        }

        with open(
            params_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                best_info,
                f,
                ensure_ascii=False,
                indent=4
            )

        best_param_records.append({
            "Model": model_name,
            "Outer_Fold": outer_fold,
            "Best_Inner_Exact": best_inner_score,
            "N_Trials": len(study.trials),
            "Best_Params": json.dumps(
                best_params,
                ensure_ascii=False
            )
        })

        # ----------------------------------------------------
        # OUTER EVALUATION
        # ----------------------------------------------------

        # Nếu đã lưu đầy đủ kết quả Outer Fold,
        # sử dụng checkpoint để tránh fit lại
        if (
            result_path.exists()
            and label_path.exists()
            and pred_path.exists()
        ):

            outer_metric_records.extend(
                lib.pd.read_csv(
                    result_path
                ).to_dict("records")
            )

            outer_label_records.extend(
                lib.pd.read_csv(
                    label_path
                ).to_dict("records")
            )

            outer_prediction_frames.append(
                lib.pd.read_csv(
                    pred_path,
                    dtype={id_col: str}
                )
            )

            print("Đã tải Outer checkpoint.")
            continue

        train_idx = outer_info["train_idx"]
        val_idx = outer_info["val_idx"]

        X_outer_train = X.iloc[train_idx]
        X_outer_val = X.iloc[val_idx]

        y_outer_train = y.iloc[train_idx]
        y_outer_val = y.iloc[val_idx]

        # Fit lại trên toàn bộ Outer Training
        best_model = create_model(
            model_name,
            best_params
        )

        best_model.fit(
            X_outer_train,
            y_outer_train
        )

        # Đánh giá Outer Validation
        y_proba = predict_positive_proba(
            best_model,
            X_outer_val
        )

        y_pred = predict_top4(y_proba)

        y_true = y_outer_val.to_numpy(
            dtype=int
        )

        assert y_pred.shape == y_true.shape
        assert (y_pred.sum(axis=1) == 4).all()

        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        metric_record = {
            "Model": model_name,
            "Outer_Fold": outer_fold,
            "Best_Inner_Exact": best_inner_score,
            "Exact_Match_Accuracy": accuracy_score(
                y_true,
                y_pred
            ),
            "Macro_F1": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            "Micro_F1": f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),
            "Hamming_Loss": hamming_loss(
                y_true,
                y_pred
            )
        }

        label_scores = f1_score(
            y_true,
            y_pred,
            average=None,
            zero_division=0
        )

        label_rows = [
            {
                "Model": model_name,
                "Outer_Fold": outer_fold,
                "Nhãn": label,
                "F1": float(score)
            }
            for label, score in zip(
                choice_cols,
                label_scores
            )
        ]

        # ----------------------------------------------------
        # SAVE PREDICTIONS
        # ----------------------------------------------------

        pred_df = lib.pd.DataFrame({
            id_col: df_dev.iloc[
                val_idx
            ][id_col].to_numpy(),
            "Model": model_name,
            "Outer_Fold": outer_fold
        })

        for j in range(9):

            pred_df[f"true_{j+1}"] = y_true[:, j]
            pred_df[f"pred_{j+1}"] = y_pred[:, j]
            pred_df[f"proba_{j+1}"] = y_proba[:, j]

        # Checkpoint lưu sau khi Outer Validation hoàn tất
        lib.pd.DataFrame(
            [metric_record]
        ).to_csv(
            result_path,
            index=False,
            encoding="utf-8-sig"
        )

        lib.pd.DataFrame(
            label_rows
        ).to_csv(
            label_path,
            index=False,
            encoding="utf-8-sig"
        )

        pred_df.to_csv(
            pred_path,
            index=False,
            encoding="utf-8-sig"
        )

        outer_metric_records.append(
            metric_record
        )

        outer_label_records.extend(
            label_rows
        )

        outer_prediction_frames.append(
            pred_df
        )

        print(
            f"Best Inner Exact: {best_inner_score:.4f}"
        )

        print(
            "Outer Exact-match:",
            round(metric_record["Exact_Match_Accuracy"], 4)
        )

        print(
            "Outer Macro-F1:",
            round(metric_record["Macro_F1"], 4)
        )


print("\nPASS: Nested Cross-Validation hoàn thành.")

MODEL: Random Forest | OUTER FOLD: 1


  0%|          | 0/11 [00:00<?, ?it/s]

Best Inner Exact: 0.1728
Outer Exact-match: 0.1753
Outer Macro-F1: 0.4534
MODEL: Random Forest | OUTER FOLD: 2


  0%|          | 0/11 [00:00<?, ?it/s]

Best Inner Exact: 0.1701
Outer Exact-match: 0.1679
Outer Macro-F1: 0.4479
MODEL: Random Forest | OUTER FOLD: 3


  0%|          | 0/11 [00:00<?, ?it/s]

... (3 outputs left out)

  0%|          | 0/11 [00:00<?, ?it/s]

Best Inner Exact: 0.1674
Outer Exact-match: 0.1786
Outer Macro-F1: 0.4542
MODEL: XGBoost | OUTER FOLD: 1


  0%|          | 0/11 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 9: FINAL PARAMETER SELECTION ON DEVELOPMENT
# ============================================================

all_dev_idx = lib.np.arange(
    len(df_dev)
)

final_inner_folds = create_inner_folds(
    all_dev_idx,
    n_splits=N_INNER_FOLDS
)

final_records = []

for model_name in MODEL_NAMES:

    print("=" * 65)
    print(f"FINAL TUNING: {model_name}")
    print("=" * 65)

    study = get_study(
        model_name,
        outer_fold=0,
        study_suffix="final_development"
    )

    final_params, final_inner_score = optimize_study(
        study,
        model_name,
        final_inner_folds,
        total_trials=N_TRIALS
    )

    slug = model_slug(model_name)

    final_config = {
        "Model": model_name,
        "Best_Params": final_params,
        "Best_Inner_Exact": final_inner_score,
        "N_Trials": len(study.trials),
        "Selection_Data": "9000 Development students",
        "Selection_Method": "3-Fold Inner CV",
        "Objective": "Exact-match Accuracy",
        "Independent_Test_Used": False
    }

    with open(
        config_dir / f"{slug}_final_best_config.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            final_config,
            f,
            ensure_ascii=False,
            indent=4
        )

    study.trials_dataframe().to_csv(
        table_dir / f"{slug}_final_trials.csv",
        index=False,
        encoding="utf-8-sig"
    )

    final_records.append({
        "Model": model_name,
        "Best_Inner_Exact": final_inner_score,
        "N_Trials": len(study.trials),
        "Best_Params": json.dumps(
            final_params,
            ensure_ascii=False
        )
    })

    print("Best Inner Exact:", round(final_inner_score, 4))
    print("Best Parameters:", final_params)


final_summary = lib.pd.DataFrame(
    final_records
)

final_summary.to_csv(
    table_dir / "final_best_params_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

experiment_config = {
    "experiment": "17 - Hyperparameter Optimization",
    "method": "Optuna Nested Cross-Validation",
    "random_seed": RANDOM_SEED,
    "outer_cv": {
        "n_splits": N_OUTER_FOLDS,
        "source": "data/data-splits/cv_folds_v1"
    },
    "inner_cv": {
        "n_splits": N_INNER_FOLDS,
        "method": "KFold",
        "shuffle": True
    },
    "optimization": {
        "sampler": "TPESampler",
        "pruner": "MedianPruner",
        "n_trials": N_TRIALS,
        "direction": "maximize",
        "objective": PRIMARY_METRIC
    },
    "models": MODEL_NAMES,
    "features": 96,
    "targets": 9,
    "prediction_rule": "Top-4 probabilities",
    "class_weight_enabled": False,
    "oversampling_enabled": False,
    "feature_engineering_enabled": False,
    "independent_test_used": False
}

with open(
    config_dir / "hyperparameter_optimization_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nPASS: Đã lưu cấu hình cuối cùng.")
display(final_summary)